# Подключение к PostgreSQL 17
Ноутбук для подключения к серверу PostgreSQL из Python, проверки соединения, просмотра таблиц и выполнения SQL-запросов.

Перед запуском укажите параметры подключения в ячейке ниже. Пароль вводится скрыто и не сохраняется в выводе ноутбука.

## 1. Установка библиотеки

In [ ]:
%pip install -q "psycopg[binary]" pandas

## 2. Параметры подключения
- Если Jupyter запущен на том же сервере, где установлен PostgreSQL, обычно используйте `localhost`.
- Если подключаетесь с другого компьютера, укажите IP-адрес Windows Server.
- Порт PostgreSQL по умолчанию — `5432`.
- Убедитесь, что для удалённого подключения разрешён доступ в брандмауэре и настроен `pg_hba.conf`.

In [ ]:
from getpass import getpass

PG_HOST = input("Адрес сервера PostgreSQL [localhost]: ").strip() or "localhost"
PG_PORT = int(input("Порт [5432]: ").strip() or "5432")
PG_DATABASE = input("Имя базы данных [postgres]: ").strip() or "postgres"
PG_USER = input("Пользователь [postgres]: ").strip() or "postgres"
PG_PASSWORD = getpass("Пароль PostgreSQL: ")


## 3. Подключение и проверка сервера

In [ ]:
import psycopg

conn = psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DATABASE,
    user=PG_USER,
    password=PG_PASSWORD,
    connect_timeout=10
)

print("Подключение успешно!")

with conn.cursor() as cur:
    cur.execute("SELECT version(), current_database(), current_user;")
    version, database, user = cur.fetchone()

print("Версия сервера:", version)
print("Текущая база:", database)
print("Пользователь:", user)

## 4. Просмотр таблиц в текущей схеме

In [ ]:
import pandas as pd

tables = pd.read_sql_query(
    """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema NOT IN ('pg_catalog', 'information_schema')
    ORDER BY table_schema, table_name;
    """,
    conn
)
display(tables)

## 5. Выполнение SQL-запроса
Замените запрос на нужный. Для `SELECT` используйте `pd.read_sql_query()`.

In [ ]:
sql = "SELECT current_timestamp AS server_time;"
result = pd.read_sql_query(sql, conn)
display(result)

## 6. Пример безопасного запроса с параметром
Передавайте значения отдельно от SQL-текста, а не вставляйте их через f-строки.

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT %s::text AS message;", ("Привет, PostgreSQL!",))
    print(cur.fetchone()[0])

## 7. Закрытие соединения

In [ ]:
conn.close()
print("Соединение закрыто.")

### Если подключение не работает
- `connection refused` — проверьте, запущена ли служба PostgreSQL и слушает ли сервер нужный порт.
- `password authentication failed` — проверьте имя пользователя и пароль.
- `no pg_hba.conf entry` — добавьте разрешающее правило в `pg_hba.conf` для IP клиента и перезапустите/перезагрузите конфигурацию PostgreSQL.
- При удалённом подключении проверьте `listen_addresses` в `postgresql.conf` и входящее правило TCP для порта `5432` в Windows Firewall.

Не открывайте порт PostgreSQL всему интернету. Для удалённого доступа предпочтительнее VPN или ограничение по доверенным IP.